# Genned: Phase 3 fine-tuning on Kaggle (accuracy + everyday edits + universal attacks)

> **Open a fresh copy from GitHub** (Kaggle: *File → Import Notebook → GitHub → as791/genned →
> notebooks/robust_finetune_kaggle.ipynb*). The setup cell refuses to run older code.

What this trains (internal-docs/MODEL.md "Phase 3"), for both of the app's models:
- **Accuracy and everyday edits:** 60% of training images get a random edit (noise, filters, rescaling, crops,
  rotation, WebP, screenshots, chains), on top of the newest open-weights generators.
- **Universal attacks:** two shared perturbations (one pushes AI images toward "real", one real images toward "AI")
  are learned *during* training and the model is trained against them (universal adversarial training, 8/255).

**Data.** Training: OpenFake's *train* split (older generators, LAION/Pexels photos) plus Defactify's train split.
Images from proprietary generators (Midjourney, DALL·E, GPT-Image, Imagen, Ideogram, Grok, Recraft, Seedream,
Nano Banana, Veo, Sora, ...) are **excluded from training**: their subsets are non-commercial and the trained weights
ship in the app. The benchmarks then test on OpenFake's *test* split, which holds the newest (2025–26) generators the
models never saw.

**Safety rails (unchanged from Phase 2b):** validation every half epoch; training stops if clean AUC collapses; a
checkpoint is saved and uploaded only if clean AUC stays within 0.01 and clean accuracy within 5 points of the start.

**Before you start (Kaggle):**
1. *Settings → Accelerator → GPU T4 x2* (one GPU is used) and *Internet → on*.
2. *Add-ons → Secrets*: **`HF_TOKEN`** (write scope; the account has accepted the gated `Dafilab/ai-image-detector` terms).
3. *Run all.* Data download about 10–20 min; Community Forensics about 30–45 min; EfficientNet-B4 about 1.5–2.5 h.
   Everything fits one 12-hour session.
4. When it finishes, copy the last cell's output to Claude.

In [ ]:
!nvidia-smi || echo 'No GPU: turn on the GPU accelerator before continuing.'

In [ ]:
import os, subprocess, shutil

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata  # Colab also works, with less time
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "Add an HF_TOKEN secret (write scope) first - see the first cell."
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
DATA = "/tmp/genned-data" if os.path.isdir("/kaggle") else os.path.join(WORK, "data")  # Kaggle: local disk, not the 20 GB output dir
os.makedirs(DATA, exist_ok=True)
os.chdir(WORK)
if os.path.isdir("genned"):
    shutil.rmtree("genned")  # always start from the latest code
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/as791/genned.git", "genned"], check=True)
os.chdir("genned")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert 'RECIPE_VERSION = "phase3-v1"' in open("tools/adv_finetune.py").read(), "Old code: re-import this notebook from GitHub."
print("Phase 3 code confirmed. Data goes to", DATA)

In [ ]:
!pip install -q "timm>=1.0,<2" "huggingface_hub>=0.23,<1" "datasets>=2.19,<5" "onnxruntime>=1.18,<2" "pillow>=10,<12"
from huggingface_hub import HfApi
HF_REPO = HfApi().whoami()["name"] + "/genned-robust"
print("Weights will be uploaded (privately) to", HF_REPO)

## Training and validation data
`--strict-split` makes a download fail rather than fall back to the test split the benchmarks use.
`PROPRIETARY` lists generators whose images are never used for training or checkpoint selection.

In [ ]:
OPENFAKE_PER_CLASS = 4000   # per class from OpenFake train (lower it if short on time or disk)
DEFACTIFY_PER_CLASS = 2000  # per class from Defactify train
PROPRIETARY = ("midjourney|dall-?e|gpt-image|imagen|ideogram|grok|recraft|seedream|nano-banana|mystic|"
               "veo|sora|kling|firefly|runway|luma|hailuo|ernie|seededit|gemini")

!python tools/fetch_eval_data.py --dataset ComplexDataLab/OpenFake --config core \
    --split train --strict-split --out {DATA}/train/openfake --per-class {OPENFAKE_PER_CLASS} \
    --label-col label --ai-values fake --real-values real --generator-col model \
    --exclude-generators "{PROPRIETARY}" --max-per-generator 300 --max-scan 200000
!python tools/fetch_eval_data.py --dataset Rajarshi-Roy-research/Defactify_Image_Dataset \
    --split train --strict-split --out {DATA}/train/defactify --per-class {DEFACTIFY_PER_CLASS} --max-scan 25000 \
    --label-col Label_A --ai-values 1 --real-values 0 \
    --generator-col Label_B --generator-names real,sd21,sdxl,sd3,dalle3,midjourney6
!python tools/fetch_eval_data.py --dataset ComplexDataLab/OpenFake --config core \
    --split validation --strict-split --out {DATA}/val/openfake --per-class 200 \
    --label-col label --ai-values fake --real-values real --generator-col model \
    --exclude-generators "{PROPRIETARY}" --max-per-generator 20 --max-scan 20000
!du -sh {DATA}/train/* {DATA}/val/*

## 1. Community Forensics ViT-S 224 (about 30–45 min)
Recipe settings come from the script's defaults; only the objective, data and tag are set here.

In [ ]:
TRAIN = f"{DATA}/train/openfake {DATA}/train/defactify"
!python tools/adv_finetune.py --model commfor --objective aug+uat \
    --train-data {TRAIN} --val-data {DATA}/val/openfake \
    --per-class {OPENFAKE_PER_CLASS} --val-per-class 200 --workers 4 --tag p3 --out runs --push-to-hub {HF_REPO}

## 2. Dafilab EfficientNet-B4 (about 1.5–2.5 h)
Both models are trained, because the app's score is the ensemble of the two.

In [ ]:
!python tools/adv_finetune.py --model bundled --objective aug+uat \
    --train-data {TRAIN} --val-data {DATA}/val/openfake \
    --per-class {OPENFAKE_PER_CLASS} --val-per-class 200 --workers 4 --tag p3 --out runs --push-to-hub {HF_REPO}

In [ ]:
import glob, json
for path in sorted(glob.glob("runs/*-train-log-p3.json")):
    log = json.load(open(path))
    b = log["base"]
    print(f"{log['model']} ({log['objective']}, {log['train_images']} training images): "
          f"clean gate met: {log['meets_clean_gate']}, stopped: {log['stopped']}")
    print(f"  start: clean AUC {b['clean_auc']:.3f}, edited AUC mean {b['edit_auc_mean']:.3f} / min {b['edit_auc_min']:.3f}, "
          f"universal robust {b['uap_robust_acc_8']:.1%}")
    for e in log["evaluations"]:
        print(f"  epoch {e['epoch']}: eps {e['eps_255']}/255: clean AUC {e['clean_auc']:.3f}, acc {e['clean_acc']:.1%}, "
              f"edited AUC {e['edit_auc_mean']:.3f} / {e['edit_auc_min']:.3f}, universal robust {e['uap_robust_acc_8']:.1%}"
              f"{'' if e['eligible'] else '  (not eligible)'}")
    if log["selected"]:
        print(f"  -> saved epoch {log['selected']['epoch']} as {log['model']}-robust-p3.pt")
print()
print("Done. Paste this summary to Claude.")